In [57]:
import pandas as pd

def top_entailment_per_target(df, entail = "ENTAILMENT"):
    """
    For each target, select the row with the highest 'entailment' score.
    
    Parameters:
        df (pd.DataFrame): DataFrame with columns including 'target' and 'entailment'.
        
    Returns:
        pd.DataFrame: DataFrame with one row per target, having the highest entailment score.
    """
    # For each target, find index of row with max entailment
    idx = df.groupby('target')[entail].idxmax(axis=0)
    return df.loc[idx].reset_index(drop=True)

# Example usage
# data = {
#     'textid': ['greeting', 'question', 'greeting', 'question','greeting', 'question'],
#     'target': [0, 0, 1, 1,2,2],
#     'model': ['huggingface/distilbert-base-uncased-finetuned-mnli']*6,
#     'tokenizer': ['huggingface/distilbert-base-uncased-finetuned-mnli']*6,
#     'predicted': ['entailment', 'entailment', 'contradiction', 'contradiction', 'contradiction', 'contradiction'],
#     'prob': [0.8553178310394287, 0.8553178310394287, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032],
#     'entailment': [0.8553178310394287, 0.8553178310394287, 0.31023797392845154, 0.31023797392845154, 0.31023797392845154, 0.31023797392845154],
#     'neutral': [0.13772304356098175, 0.13772304356098175, 0.3048897087574005, 0.3048897087574005, 0.3048897087574005, 0.3048897087574005],
#     'contradiction': [0.006959038320928812, 0.006959038320928812, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032]
# }

data = pd.read_csv("predictions/Copy_Backup/Shared_BaseTest_predictions.tsv",sep="\t")

# df = pd.DataFrame(data)
top_df = top_entailment_per_target(data)
display(top_df)

#  {0:"World" ,1:"Sports",2:"Business",3:"Sci/Tech"}



/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_29467/2281850440.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')[entail].idxmax(axis=0)


,textid,target,model,tokenizer,predicted,prob,CONTRADICTION,NEUTRAL,ENTAILMENT
0,1,0,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.498680,0.427861,0.498680,0.073459
1,1,1,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.884839,0.884839,0.052157,0.063004
2,3,2,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.832043,0.145391,0.832043,0.022566
3,1,3,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.916654,0.056780,0.916654,0.026565
4,2,4,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.828863,0.126318,0.828863,0.044819
...,...,...,...,...,...,...,...,...,...
7595,1,7595,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.823755,0.823755,0.137614,0.038632
7596,2,7596,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.582558,0.402302,0.582558,0.015140
7597,2,7597,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.914520,0.019696,0.914520,0.065784
7598,3,7598,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.680466,0.164015,0.680466,0.155519


In [27]:
data

,textid,target,model,tokenizer,predicted,prob,CONTRADICTION,NEUTRAL,ENTAILMENT
0,0,0,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.948134,0.948134,0.045721,0.006145
1,1,0,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.498680,0.427861,0.498680,0.073459
2,2,0,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.695733,0.695733,0.270892,0.033375
3,3,0,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.717915,0.258338,0.717915,0.023747
4,0,1,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.974294,0.974294,0.024323,0.001383
...,...,...,...,...,...,...,...,...,...
30395,3,7598,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.680466,0.164015,0.680466,0.155519
30396,0,7599,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.717300,0.717300,0.268941,0.013759
30397,1,7599,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.872717,0.872717,0.123776,0.003506
30398,2,7599,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.533551,0.453124,0.533551,0.013326


In [58]:
import pandas as pd
import numpy as np


def get_positive_example(df, percent=0.1, score_cols=['entailment', 'neutral', 'contradiction']):
    """
    From the top entailment per target, select the top X% most confident examples 
    based on the delta between the top and second-highest MNLI scores.
    
    Parameters:
        df (pd.DataFrame): DataFrame with MNLI score columns.
        percent (float): Fraction of top examples to return (0 < percent <= 1)
        score_cols (list): Names of MNLI score columns to consider
        
    Returns:
        pd.DataFrame: DataFrame with top X% most confident examples.
    """
    # First, select top entailment per target

    
    # Extract MNLI score values
    top_df = top_entailment_per_target(data)
    scores = df.groupby("target")["ENTAILMENT"]
    # print(  scores)
    #Maximum values per row:
    largeset_score = scores.max()
    # make it so that 
    # all value before -2 are less thanit and all value after it are greater, so we specify that there is only index -1 greater than it
    second_largest = scores.apply(lambda x: np.partition(x, -2)[-2])
    
    # print("largest\n")
    # print(largeset_score)
    # print("second largest\n")
    # print( second_largest )
    
    delta =  largeset_score-second_largest 
    # print("delta\n")
    # print( delta )
    
    # this gets us the value on the top
    
    top_df["delta"]= delta
    
    display(top_df)
    
    # the smaller index the greater
    df_sorted_delta = top_df.sort_values(by='delta', ascending=False)
    
    top_percent_df = df_sorted_delta[:int(len(df_sorted_delta)*percent)]
    
    top_percent_df["target"] =  int(len(df_sorted_delta)*percent) * [score_cols[0]]

    return top_percent_df
    

    


# Only MNLI scores matter
mnli_labels = ['ENTAILMENT', 'NEUTRAL', 'CONTRADICTION']

# mnli_labels =['entailment', 'neutral', 'contradiction']



# Get top 10% most confident rows

print("positive\n")
display(get_positive_example(data, percent=0.01, score_cols=mnli_labels))

positive



/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_29467/2281850440.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')[entail].idxmax(axis=0)


,textid,target,model,tokenizer,predicted,prob,CONTRADICTION,NEUTRAL,ENTAILMENT,delta
0,1,0,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.498680,0.427861,0.498680,0.073459,0.040084
1,1,1,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.884839,0.884839,0.052157,0.063004,0.015977
2,3,2,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.832043,0.145391,0.832043,0.022566,0.003208
3,1,3,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.916654,0.056780,0.916654,0.026565,0.010996
4,2,4,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.828863,0.126318,0.828863,0.044819,0.009180
...,...,...,...,...,...,...,...,...,...,...
7595,1,7595,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.823755,0.823755,0.137614,0.038632,0.014344
7596,2,7596,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.582558,0.402302,0.582558,0.015140,0.006711
7597,2,7597,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.914520,0.019696,0.914520,0.065784,0.062196
7598,3,7598,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.680466,0.164015,0.680466,0.155519,0.074618


/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_29467/4278644880.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_percent_df["target"] =  int(len(df_sorted_delta)*percent) * [score_cols[0]]


,textid,target,model,tokenizer,predicted,prob,CONTRADICTION,NEUTRAL,ENTAILMENT,delta
7586,1,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.958399,0.001563,0.040038,0.958399,0.926439
5480,0,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.905279,0.015529,0.079192,0.905279,0.898234
4011,1,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.935952,0.002265,0.061782,0.935952,0.886754
4513,1,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.968252,0.001655,0.030093,0.968252,0.879723
5713,1,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.930366,0.002373,0.067260,0.930366,0.876182
...,...,...,...,...,...,...,...,...,...,...
3912,0,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.766721,0.033962,0.199317,0.766721,0.682999
3972,2,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.697208,0.032673,0.270119,0.697208,0.681805
2345,3,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.709917,0.021731,0.268352,0.709917,0.681364
4903,0,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.721813,0.046719,0.231468,0.721813,0.678806


In [ ]:
#  {0:"World" ,1:"Sports",2:"Business",3:"Sci/Tech"}

In [ ]:
# this is just redefining the same function without the print so it is less annoying 

def get_positive_example(df, percent=0.1, score_cols=['entailment', 'neutral', 'contradiction']):
    """
    From the top entailment per target, select the top X% most confident examples 
    based on the delta between the top and second-highest MNLI scores.
    
    Parameters:
        df (pd.DataFrame): DataFrame with MNLI score columns.
        percent (float): Fraction of top examples to return (0 < percent <= 1)
        score_cols (list): Names of MNLI score columns to consider
        
    Returns:
        pd.DataFrame: DataFrame with top X% most confident examples.
    """
    # First, select top entailment per target

    
    # Extract MNLI score values
    top_df = top_entailment_per_target(data)
    scores = df.groupby("target")["ENTAILMENT"]
    # print(  scores)
    #Maximum values per row:
    largeset_score = scores.max()
    # make it so that 
    # all value before -2 are less thanit and all value after it are greater, so we specify that there is only index -1 greater than it
    second_largest = scores.apply(lambda x: np.partition(x, -2)[-2])
    
    # print("largest\n")
    # print(largeset_score)
    # print("second largest\n")
    # print( second_largest )
    
    delta =  largeset_score-second_largest 
    # print("delta\n")
    # print( delta )
    
    # this gets us the value on the top
    
    top_df["delta"]= delta
    
    # the smaller index the greater
    df_sorted_delta = top_df.sort_values(by='delta', ascending=False)
    
    top_percent_df = df_sorted_delta[:int(len(df_sorted_delta)*percent)]
    
    top_percent_df["target"] =  int(len(df_sorted_delta)*percent) * [score_cols[0]]

    return top_percent_df
    

    

In [ ]:
    
import random

def get_negative_random(data,percent=0.1,contra = "CONTRADICTION"):
    """
    For each entailment pair, generate a negative example by replacing the class
    in the hypothesis with a random different class, and assign the contradict label.
    
    Parameters:
    
       Positve df
        
    Returns:
       Full finetuning dataset
    """

    pos = get_positive_example(data,percent=percent).dropna().reset_index(drop=True)
    sample_space = set(pos["textid"].unique())
    for idx in range(len(pos)):
        pos.loc[idx, "target"] = contra 
        event1 =  {pos.loc[idx, "textid"]}
        pos.loc[idx, "textid"] = int(random.sample(list(sample_space - event1), 1)[0])
        mapping_mask =  {0:"World" ,1:"Sports",2:"Business",3:"Sci/Tech"}
        label_type = mapping_mask[int(pos.loc[idx, "textid"])]
        pos.loc[idx, "pair"] = f"This example is {label_type}"
    
    return pos
        

        # # pos["predicted"] = pd.sample([])
        # mapping_mask =  {0:"World" ,1:"Sports",2:"Business",3:"Sci/Tech"}
        # text_classfication_true["predicted"] =  Max_entailment["textid"].map(mapping_mask)

    # pd.concat([])
    



get_negative_random(data,percent=0.01)


/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_29467/2882925959.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_percent_df["target"] =  int(len(df_sorted_delta)*percent) * [score_cols[0]]


,textid,target,model,tokenizer,predicted,prob,CONTRADICTION,NEUTRAL,ENTAILMENT,delta,pair
0,2,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.958399,0.001563,0.040038,0.958399,0.926439,This example is Business
1,3,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.905279,0.015529,0.079192,0.905279,0.898234,This example is Sci/Tech
2,3,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.935952,0.002265,0.061782,0.935952,0.886754,This example is Sci/Tech
3,2,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.968252,0.001655,0.030093,0.968252,0.879723,This example is Business
4,2,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.930366,0.002373,0.067260,0.930366,0.876182,This example is Business
...,...,...,...,...,...,...,...,...,...,...,...
71,1,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.766721,0.033962,0.199317,0.766721,0.682999,This example is Sports
72,0,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.697208,0.032673,0.270119,0.697208,0.681805,This example is World
73,0,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.709917,0.021731,0.268352,0.709917,0.681364,This example is World
74,3,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.721813,0.046719,0.231468,0.721813,0.678806,This example is Sci/Tech


In [65]:
originl_train_data = pd.read_csv("NLP_dataset/MNLI_formatted/Shared_Test_MNLI.tsv",sep="\t")
# display(originl_train_data)
originl_train_data
originl_train_data.loc[7586*4,"pair"]

'This example is World'

In [67]:

def create_fintune_data(data,originl_train_data,percent,num_label = 4):
    
    pos = get_positive_example(data,percent)
    neg = get_negative_random(data,percent)
    
    org_pos = pos.index 
    mnli_index_pos = org_pos * num_label 
    mapping_mask =  {0:"World" ,1:"Sports",2:"Business",3:"Sci/Tech"}
    # Map textid -> label and format each as a string
    
    # apply a mask and create template
    pos_pairs = pos["textid"].map(mapping_mask).apply(lambda x: f"This example is {x}")

    # Combine with neg pairs
    # 
    pair_series = pd.concat([pos_pairs, neg["pair"]], ignore_index=True)


    template = pos["textid"].map(mapping_mask),neg["pair"]
    output = pd.DataFrame({
        
        "textid": range(2 * len(pos)),
        "pair": pair_series,
        "text":  pd.concat([originl_train_data.loc[mnli_index_pos,"text"], originl_train_data.loc[mnli_index_pos,"text"],],ignore_index=True),
        "label": pd.concat([pd.Series(len(pos)*["ENTAILMENT"]), neg["target"]],ignore_index=True)
        
        
    })
    
    return output

# pd.read_csv("NLP_dataset/MNLI_formatted/Shared_Test_MNLI.tsv",sep="\t")
# data = pd.read_csv("predictions/Copy_Backup/Shared_BaseTest_predictions.tsv",sep="\t")

    
output = create_fintune_data(data,originl_train_data,0.01)


output.to_csv("test.tsv",sep="\t",index=False)
display(output)


/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_29467/2882925959.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_percent_df["target"] =  int(len(df_sorted_delta)*percent) * [score_cols[0]]


,textid,pair,text,label
0,0,This example is Sports,"Miami, FL (Sports Network) - Shaquille O #39;N...",ENTAILMENT
1,1,This example is World,PC World #39;s first tests of NVidia #39;s jus...,ENTAILMENT
2,2,This example is Sports,INDIANAPOLIS (Sports Network) - Peyton Mannin...,ENTAILMENT
3,3,This example is Sports,"Baltimore, MD (Sports Network) - Baltimore Rav...",ENTAILMENT
4,4,This example is Sports,"Jacksonville, FL (Sports Network) - David Garr...",ENTAILMENT
...,...,...,...,...
147,147,This example is Sci/Tech,"This time, world leaders and their people cann...",CONTRADICTION
148,148,This example is Sports,"Seven years of Pedro. Went by quickly, huh? Se...",CONTRADICTION
149,149,This example is Business,"&lt;a href=""http://arstechnica.com/news/posts/...",CONTRADICTION
150,150,This example is Sci/Tech,World number one Vijay Singh shot a four-under...,CONTRADICTION


In [18]:
data.loc[data["ENTAILMENT"].idxmax()]

textid                                     1
target                                  4513
model            microsoft/deberta-base-mnli
tokenizer        microsoft/deberta-base-mnli
predicted                         ENTAILMENT
prob                                0.968252
CONTRADICTION                       0.001655
NEUTRAL                             0.030093
ENTAILMENT                          0.968252
Name: 18053, dtype: object